In [1]:
import pandas as pd
import pickle
import re
import pymorphy3

In [2]:
with open('stopwords_ua.txt', 'r', encoding='utf-8') as f:
    context = f.read()
    stopwords = context.split()
print(stopwords[:30])

['а', 'аби', 'абиде', 'абиким', 'абикого', 'абиколи', 'абикому', 'абикуди', 'абихто', 'абичий', 'абичийого', 'абичийому', 'абичим', 'абичию', 'абичия', 'абичиє', 'абичиєму', 'абичиєю', 'абичиєї', 'абичиї', 'абичиїй', 'абичиїм', 'абичиїми', 'абичиїх', 'абичого', 'абичому', 'абищо', 'абияка', 'абияке', 'абиякий']


In [3]:
morph = pymorphy3.MorphAnalyzer(lang='uk')

def filtering(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r'[^\w\s]', '', sentence)
    words = sentence.split()
    clean_text = [word for word in words if word not in stopwords]
    lemmatized_text = [morph.parse(word)[0].normal_form for word in clean_text]
    return ' '.join(lemmatized_text)

print(filtering(sentence="Я сьогодні йшов на пари, але по дорозі зустрів подругу та поїхав назад додому. Пари відмінили, і я зайнявся домашнім завданням готуючись до наступного дня!"))

йти пара дорога зустріти подруга поїхати додому пара відмінити зайнятися домашній завдання готуючись наступний день


In [4]:
#detector model
try:
    with open('DETECTOR/fake_true_vectorizer.pkl', 'rb') as f:
        fake_vectorizer = pickle.load(f)
    with open('DETECTOR/model_detector.pkl', 'rb') as f:
        fake_model = pickle.load(f)
    print("Files opened succesfully!")
except Exception as e:
    print(f"An unexpected error occured: {e}")

Files opened succesfully!


In [5]:
#theme model
try:
    with open('THEME/theme_vectorizer.pkl', 'rb') as f:
        theme_vectorizer = pickle.load(f)
    with open('THEME/theme_model.pkl', 'rb') as f:
        theme_model = pickle.load(f)
    print("Files opened succesfully!")
except Exception as e:
    print(f"An unexpected error occured: {e}")

Files opened succesfully!


In [6]:
def analyze_news(text):
    processed_text = filtering(text)
    fake_vector = fake_vectorizer.transform([processed_text])
    is_fake = fake_model.predict(fake_vector)
    prob_model = fake_model.predict_proba(fake_vector)
    if prob_model[0][1] > 0.5:
        probability = float(prob_model[0][1])
        label = "правда"
    else:
        probability = float(prob_model[0][0])
        label = "фейк"
    theme_vector = theme_vectorizer.transform([processed_text])
    theme = theme_model.predict(theme_vector)[0]
    return {
        'theme': theme,
        'is_fake': bool(is_fake),
        'label': label,
        'probability': round(probability, 4)
    }


In [7]:
import time
def print_analyzed_news(text):
    analyzed_text = analyze_news(text)
    print("="*75)
    print(f"Текст: \n{text}")
    print("="*75, '\n')
    
    print("Аналіз...")
    
    time.sleep(0.5)
    
    print("="*75)    
    print(f"Тема: {analyzed_text['theme'].upper()}")
    if analyzed_text['is_fake']:
        print(f"Статус -> {analyzed_text['label']}!")
    else:
        print(f"Статус -> {analyzed_text['label']}!")
    print("="*75)

In [8]:
text = "Депутати ВРУ пропонують законопроект, який передасть наукові установи, лікарні та школи приватним власникам, що залишить українців без науки, медицини та освіти"
print_analyzed_news(text)

Текст: 
Депутати ВРУ пропонують законопроект, який передасть наукові установи, лікарні та школи приватним власникам, що залишить українців без науки, медицини та освіти

Аналіз...
Тема: ПОЛІТИКА
Статус -> фейк!
